In [1]:
import os
import pandas as pd
import numpy as np
import datetime as dt
import pytz
from tqdm import tqdm

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2025-01-31 17:48:07.246593


#### Constants

In [3]:
# project
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')

# task
str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'

Project: 20241112-simple-model-test
Task: 07_join_targets


#### Make output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Import data

In [5]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/06_join_tsp_data/{str_filename}'
df = pd.read_parquet(str_uri)
# show
df

CPU times: user 9.18 s, sys: 3.81 s, total: 13 s
Wall time: 6.96 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,fltapproveddowntotal__app,payment__app,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,totaldebt__app
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,2000.0,344.95,0.494484,0.142150,0,1.150000,auto,1,2012-05-30 10:50:45.437,855.00
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1000.0,598.14,0.363221,0.102956,1,1.000289,auto,1,2018-12-21 12:37:32.413,1512.06
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0.0,418.11,0.414040,0.111234,1,1.149861,auto,1,2010-01-25 15:07:34.910,1138.20
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0.0,679.78,0.380767,0.125498,0,0.993292,suv,1,2015-05-04 16:04:37.317,1382.71
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0.0,399.64,0.418160,0.083485,1,0.945898,auto,1,2012-12-27 10:14:39.477,1602.08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,1000.0,848.97,0.319642,0.104996,1,1.096184,suv,1,2016-08-30 13:05:21.300,1735.57
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,500.0,699.46,0.319354,0.119353,1,0.955559,suv,1,2024-09-06 11:26:41.533,1172.09
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,0.0,445.28,0.286484,0.086483,0,1.152536,auto,1,2018-09-06 15:55:22.703,1029.75
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,2200.0,586.59,0.451532,0.115865,0,0.934501,auto,0,2013-06-28 09:08:19.850,1699.38


#### Import targets - classification

In [6]:
str_filename = 'df_targets.gzip'
str_uri = f's3://{str_project}/01_get_targets/01_classification/{str_filename}'
df_tmp = pd.read_parquet(str_uri)
# rename
dict_rename = {
    'bigAccountId': 'accountid',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# set dtype
df_tmp['accountid'] = df_tmp['accountid'].astype(int)
# show
df_tmp

,accountid,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date
0,6,0,0,0,0,0,2025-01-31 10:43:13.940299
1,25,1,0,0,1,1,2025-01-31 10:43:13.940299
2,73,0,0,0,0,1,2025-01-31 10:43:13.940299
3,82,1,1,1,1,1,2025-01-31 10:43:13.940299
4,122,0,0,0,1,1,2025-01-31 10:43:13.940299
...,...,...,...,...,...,...,...
342779,8602995,0,0,0,0,0,2025-01-31 10:43:13.940299
342780,8603355,0,0,0,0,0,2025-01-31 10:43:13.940299
342781,8604347,0,0,0,0,0,2025-01-31 10:43:13.940299
342782,8607136,0,0,0,0,0,2025-01-31 10:43:13.940299


#### Join

In [7]:
%%time

df = pd.merge(
    left=df,
    right=df_tmp,
    on='accountid',
    how='inner',
)
# save memory
del df_tmp
# show
df

CPU times: user 450 ms, sys: 423 ms, total: 873 ms
Wall time: 1.05 s


,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,strvehicletype__app,bitgap__app,dealerstampcreation__app,totaldebt__app,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,auto,1,2012-05-30 10:50:45.437,855.00,0,0,0,0,0,2025-01-31 10:43:13.940299
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,auto,1,2018-12-21 12:37:32.413,1512.06,1,0,1,1,1,2025-01-31 10:43:13.940299
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,auto,1,2010-01-25 15:07:34.910,1138.20,0,0,0,0,0,2025-01-31 10:43:13.940299
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,suv,1,2015-05-04 16:04:37.317,1382.71,0,0,1,1,1,2025-01-31 10:43:13.940299
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,auto,1,2012-12-27 10:14:39.477,1602.08,0,0,0,0,0,2025-01-31 10:43:13.940299
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,suv,1,2016-08-30 13:05:21.300,1735.57,1,0,0,0,0,2025-01-31 10:43:13.940299
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,suv,1,2024-09-06 11:26:41.533,1172.09,1,1,1,1,0,2025-01-31 10:43:13.940299
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,auto,1,2018-09-06 15:55:22.703,1029.75,1,0,0,0,0,2025-01-31 10:43:13.940299
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,auto,0,2013-06-28 09:08:19.850,1699.38,0,0,0,0,0,2025-01-31 10:43:13.940299


#### Get days on books

In [8]:
df['days_on_books'] = (df['run_date'] - df['dtmfunded__app']).dt.days
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,bitgap__app,dealerstampcreation__app,totaldebt__app,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date,days_on_books
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,1,2012-05-30 10:50:45.437,855.00,0,0,0,0,0,2025-01-31 10:43:13.940299,1285
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1,2018-12-21 12:37:32.413,1512.06,1,0,1,1,1,2025-01-31 10:43:13.940299,1285
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,1,2010-01-25 15:07:34.910,1138.20,0,0,0,0,0,2025-01-31 10:43:13.940299,1285
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,1,2015-05-04 16:04:37.317,1382.71,0,0,1,1,1,2025-01-31 10:43:13.940299,1284
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,1,2012-12-27 10:14:39.477,1602.08,0,0,0,0,0,2025-01-31 10:43:13.940299,1283
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,1,2016-08-30 13:05:21.300,1735.57,1,0,0,0,0,2025-01-31 10:43:13.940299,67
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,1,2024-09-06 11:26:41.533,1172.09,1,1,1,1,0,2025-01-31 10:43:13.940299,67
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,1,2018-09-06 15:55:22.703,1029.75,1,0,0,0,0,2025-01-31 10:43:13.940299,67
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,0,2013-06-28 09:08:19.850,1699.38,0,0,0,0,0,2025-01-31 10:43:13.940299,67


#### Import targets - regression

In [9]:
str_filename = 'df_loss.gzip'
str_uri = f's3://{str_project}/01_get_targets/02_regression/{str_filename}'
df_tmp = pd.read_parquet(str_uri)
# rename
dict_rename = {
    'bigAccountId': 'accountid',
}
df_tmp.rename(columns=dict_rename, inplace=True)
# set dtype
df_tmp['accountid'] = df_tmp['accountid'].astype(int)
# show
df_tmp

,accountid,dtmFunded,fltNetChgOff,MonthEndDate,bktype,dtmFunded_first,MonthEndDate_first,years,months,months_tmp,months_on_books
index,,,,,,,,,,,
0,370217,2006-08-30 17:12:22.000,6796.08,2019-12-31,nobk,2006-08-01,2019-12-01,13,156,4,160
1,306072,2005-10-11 17:27:40.000,17702.55,2019-12-31,nobk,2005-10-01,2019-12-01,14,168,2,170
2,245270,2004-10-13 00:00:00.000,4755.88,2019-12-31,nobk,2004-10-01,2019-12-01,15,180,2,182
3,196070,2002-07-29 00:00:00.000,9161.37,2019-12-31,nobk,2002-07-01,2019-12-01,17,204,5,209
4,239071,2004-08-26 00:00:00.000,15970.30,2019-12-31,nobk,2004-08-01,2019-12-01,15,180,4,184
...,...,...,...,...,...,...,...,...,...,...,...
137593,5765303,2021-11-12 13:19:13.927,778.48,2024-11-30,bk,2021-11-01,2024-11-01,3,36,0,36
137594,5774350,2021-10-08 14:56:53.467,657.05,2024-11-30,bk,2021-10-01,2024-11-01,3,36,1,37
137595,5790780,2021-11-09 11:03:59.943,0.00,2024-11-30,bk,2021-11-01,2024-11-01,3,36,0,36


#### 2 Months

In [10]:
df_tmp_2 = df_tmp[df_tmp['months_on_books'] == 2].copy()
dict_map = dict(zip(df_tmp_2['accountid'], df_tmp_2['fltNetChgOff']))
df['loss_at_60'] = df['accountid'].map(dict_map)
df['loss_at_60'] = df['loss_at_60'].fillna(0)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,dealerstampcreation__app,totaldebt__app,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date,days_on_books,loss_at_60
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,2012-05-30 10:50:45.437,855.00,0,0,0,0,0,2025-01-31 10:43:13.940299,1285,0.0
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,2018-12-21 12:37:32.413,1512.06,1,0,1,1,1,2025-01-31 10:43:13.940299,1285,0.0
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,2010-01-25 15:07:34.910,1138.20,0,0,0,0,0,2025-01-31 10:43:13.940299,1285,0.0
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,2015-05-04 16:04:37.317,1382.71,0,0,1,1,1,2025-01-31 10:43:13.940299,1284,0.0
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,2012-12-27 10:14:39.477,1602.08,0,0,0,0,0,2025-01-31 10:43:13.940299,1283,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,2016-08-30 13:05:21.300,1735.57,1,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,2024-09-06 11:26:41.533,1172.09,1,1,1,1,0,2025-01-31 10:43:13.940299,67,0.0
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,2018-09-06 15:55:22.703,1029.75,1,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,2013-06-28 09:08:19.850,1699.38,0,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0


#### 6 Months

In [11]:
df_tmp_2 = df_tmp[df_tmp['months_on_books'] == 6].copy()
dict_map = dict(zip(df_tmp_2['accountid'], df_tmp_2['fltNetChgOff']))
df['loss_at_180'] = df['accountid'].map(dict_map)
df['loss_at_180'] = df['loss_at_180'].fillna(0)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,totaldebt__app,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date,days_on_books,loss_at_60,loss_at_180
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,855.00,0,0,0,0,0,2025-01-31 10:43:13.940299,1285,0.0,0.0
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1512.06,1,0,1,1,1,2025-01-31 10:43:13.940299,1285,0.0,0.0
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,1138.20,0,0,0,0,0,2025-01-31 10:43:13.940299,1285,0.0,0.0
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,1382.71,0,0,1,1,1,2025-01-31 10:43:13.940299,1284,0.0,0.0
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,1602.08,0,0,0,0,0,2025-01-31 10:43:13.940299,1283,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,1735.57,1,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,1172.09,1,1,1,1,0,2025-01-31 10:43:13.940299,67,0.0,0.0
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,1029.75,1,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,1699.38,0,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0


#### 12 Months

In [12]:
df_tmp_2 = df_tmp[df_tmp['months_on_books'] == 12].copy()
dict_map = dict(zip(df_tmp_2['accountid'], df_tmp_2['fltNetChgOff']))
df['loss_at_360'] = df['accountid'].map(dict_map)
df['loss_at_360'] = df['loss_at_360'].fillna(0)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,Early_Pay_Delinquency_15_60_Flag,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date,days_on_books,loss_at_60,loss_at_180,loss_at_360
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0,0,0,0,0,2025-01-31 10:43:13.940299,1285,0.0,0.0,0.0
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,1,0,1,1,1,2025-01-31 10:43:13.940299,1285,0.0,0.0,0.0
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,0,0,0,2025-01-31 10:43:13.940299,1285,0.0,0.0,0.0
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0,0,1,1,1,2025-01-31 10:43:13.940299,1284,0.0,0.0,0.0
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0,0,0,0,0,2025-01-31 10:43:13.940299,1283,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,1,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,1,1,1,1,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,1,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,0,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0


#### 24 Months

In [13]:
df_tmp_2 = df_tmp[df_tmp['months_on_books'] == 24].copy()
dict_map = dict(zip(df_tmp_2['accountid'], df_tmp_2['fltNetChgOff']))
df['loss_at_720'] = df['accountid'].map(dict_map)
df['loss_at_720'] = df['loss_at_720'].fillna(0)
# show
df

,accountid,request_datetime,response_model_name,file_key,bitdebtor,bitdebtor__app,dealerstate__app,strdealershiptrackertype__app,strname__app,bitdealertrack__app,...,Early_Pay_Delinquency_30_90_Flag,Early_Pay_Delinquency_30_180_Flag,Early_Pay_Delinquency_30_360_Flag,Early_Pay_Delinquency_60_720_Flag,run_date,days_on_books,loss_at_60,loss_at_180,loss_at_360,loss_at_720
0,5702434,2021-07-26 16:29:29.3903686,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Iowa,Franchise,Iowa,True,...,0,0,0,0,2025-01-31 10:43:13.940299,1285,0.0,0.0,0.0,0.00
1,5714239,2021-07-26 16:39:34.1121025,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Utah,Franchise,Utah,True,...,0,1,1,1,2025-01-31 10:43:13.940299,1285,0.0,0.0,0.0,7827.16
2,5713063,2021-07-26 16:48:39.3211104,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Illinois,Franchise,Illinois,True,...,0,0,0,0,2025-01-31 10:43:13.940299,1285,0.0,0.0,0.0,0.00
3,5713732,2021-07-27 09:02:35.3300974,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Michigan,Independent,Michigan,False,...,0,1,1,1,2025-01-31 10:43:13.940299,1284,0.0,0.0,0.0,0.00
4,5715634,2021-07-27 09:18:12.2190097,Gen10,deprecated/03_pull_payloads_tbldove/df_request...,1,1,Arizona,Franchise,Arizona,False,...,0,0,0,0,2025-01-31 10:43:13.940299,1283,0.0,0.0,0.0,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94472,8420588,2024-11-26 06:16:16+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,North Carolina,Independent,North Carolina,True,...,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0,0.00
94473,8401043,2024-11-26 06:21:44+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Nevada,Franchise,Nevada,True,...,1,1,1,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0,0.00
94474,8414683,2024-11-26 06:25:09+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Virginia,Franchise,Virginia,False,...,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0,0.00
94475,8359085,2024-11-26 06:32:28+00:00,PRESTIGE-GEN-XII,01_pull_payloads/df_requests_2024-11-25.gzip,1,1,Alabama,Franchise,Alabama,True,...,0,0,0,0,2025-01-31 10:43:13.940299,67,0.0,0.0,0.0,0.00


#### Write to s3

In [14]:
%%time

str_filename = 'df.gzip'
str_uri = f's3://{str_project}/{str_task}/{str_filename}'
df.to_parquet(str_uri, compression='gzip')

CPU times: user 44.8 s, sys: 268 ms, total: 45.1 s
Wall time: 48.8 s
